In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *


df_sentiment = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfTimeComplate.pkl")
df_sentiment = df_sentiment.loc[df_sentiment["Company"]=="AAPL"]#.groupby("Date").count()
df_sentiment = df_sentiment.rename(columns={"Date":"date"})

In [ ]:
df_sentiment = df_sentiment.loc[df_sentiment["Company"]=="AAPL"]#.groupby("Date").count()

In [ ]:
df_sentiment = df_sentiment.rename(columns={"Date":"date"})

In [ ]:
df_sentiment.date.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/EventStudy/PKL/Stocks/AAPL.pkl")

In [ ]:
df_AAPL = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/Data/PKL/Stocks/AAPL.pkl")

In [ ]:
pd.merge(df_AAPL[["date","changeOverTime"]])

In [ ]:
news_data

In [ ]:
import pandas as pd
import numpy as np

# load datapd.read_csv('vix_data.csv', parse_dates=['Date'])
vix_data = df_AAPL
news_data = df_sentiment

# compute sentiment scores
news_data['sentiment_score'] = news_data['Sentiment return']#.apply(compute_sentiment)

# merge data
data = pd.merge(vix_data, news_data[['date', 'sentiment_score']], on='date')#, how='left')

# define event windows
event_dates = df_sentiment.index
# [
#     pd.to_datetime('2023-02-27'),
#     pd.to_datetime('2023-03-03'),
#     pd.to_datetime('2023-03-08')
# ]
event_windows = [
    pd.Timedelta(days=3),
    pd.Timedelta(days=5),
    pd.Timedelta(days=7)
]

# initialize results DataFrame
results = pd.DataFrame(columns=['event_date', 'event_window', 'event_return', 'benchmark_return', 'abnormal_return', 't_statistic', 'p_value'])

# loop over events
for i, event_date in enumerate(event_dates):
    # compute average returns during event window
    event_window = event_windows[i]
    event_data = data[(data['Date'] >= (event_date - event_window)) & (data['Date'] <= (event_date + event_window))]
    event_return = event_data['Close'].mean()

    # compute benchmark returns during event window
    benchmark_data = data[(data['Date'] >= (event_date - event_window - pd.Timedelta(days=30))) & (data['Date'] <= (event_date - event_window))]
    benchmark_return = benchmark_data['Close'].mean()

    # compute abnormal return
    abnormal_return = event_return - benchmark_return

    # compute standard deviation of benchmark returns
    benchmark_std = benchmark_data['Close'].std()

    # compute t-statistic
    t_statistic = abnormal_return / (benchmark_std / np.sqrt(len(benchmark_data)))

    # compute p-value
    p_value = 2 * (1 - norm.cdf(abs(t_statistic)))

    # add results to DataFrame
    results.loc[i] = [event_date, event_window, event_return, benchmark_return, abnormal_return, t_statistic, p_value]

# print results
print(results)